### Análise Exploratória das Métricas Globais de Conectividade

Este notebook apresenta a análise exploratória das métricas globais de conectividade funcional cerebral extraídas a partir de sinais de EEG em estado de repouso.

O objetivo desta etapa é:
- caracterizar estatisticamente as métricas globais,
- investigar diferenças entre condições experimentais (EO vs EC),
- fornecer subsídios para as etapas posteriores de redução de dimensionalidade e aprendizado de máquina.


### Import e carregamento

In [1]:
import pandas as pd
import numpy as np
import plotly.io as pio

import plotly.express as px
import plotly.graph_objects as go

In [2]:
pio.templates.default = "plotly_dark"

In [3]:
# Carrega dataset final
df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_dataset.shape

(108, 360)

In [4]:
df_dataset

,degree_AF3,clustering_AF3,betweenness_AF3,hub_frequency_AF3,sd_degree_AF3,bin_clustering_AF3,degree_AF4,clustering_AF4,betweenness_AF4,hub_frequency_AF4,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,0.403697,0.356960,0.005195,0.001017,0.153145,0.624913,0.418594,0.369780,0.000000,0.002542,...,False,False,False,False,True,False,True,True,False,False
1,0.454580,0.415602,0.003759,0.000000,0.159919,0.679399,0.458031,0.412871,0.008772,0.004067,...,False,False,False,False,True,False,True,True,False,False
2,0.588981,0.575718,0.000584,0.006609,0.193803,0.711421,0.571515,0.562412,0.000000,0.001525,...,False,False,False,False,True,False,True,False,True,True
3,0.454194,0.443422,0.000565,0.000508,0.180212,0.657361,0.445602,0.438874,0.000000,0.000508,...,False,False,False,False,True,False,True,False,True,True
4,0.643513,0.612853,0.000627,0.002542,0.155180,0.757100,0.643611,0.612303,0.001880,0.000000,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,0.395055,0.393771,0.002020,0.001017,0.162625,0.622700,0.404030,0.397058,0.016835,0.000000,...,False,False,False,False,True,True,False,True,False,True
104,0.474781,0.484025,0.000000,0.001017,0.194391,0.705043,0.365710,0.409217,0.000000,0.000000,...,False,False,False,False,True,False,True,True,False,False
105,0.419353,0.433576,0.008475,0.000000,0.198661,0.660947,0.362828,0.382282,0.001695,0.001017,...,False,False,False,False,True,False,True,True,False,False
106,0.581148,0.566322,0.011105,0.001017,0.193871,0.732719,0.566560,0.559494,0.001753,0.000000,...,False,False,False,False,True,False,True,True,False,True


In [5]:
df_dataset['condition'].value_counts()

condition
EC    54
EO    54
Name: count, dtype: int64

### Seleção das métricas globais

“As métricas globais representam propriedades agregadas da rede funcional cerebral, independententes de regiões específicas, refletindo o nível geral de integração e variabilidade da conectividade.”

In [6]:
import pandas as pd

GLOBAL_FEATURES = [
    "edges_mean",
    "edges_std",
    "edges_cv",
]

id_cols = ["subject_id", "condition"]

if "df_dataset" not in globals():
    df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_global = df_dataset[id_cols + GLOBAL_FEATURES]

df_global.head()

,subject_id,condition,edges_mean,edges_std,edges_cv
0,sub-010069,EC,634.610574,149.253039,0.235188
1,sub-010069,EO,766.682257,198.573876,0.259004
2,sub-010070,EC,1064.812913,237.521762,0.223064
3,sub-010070,EO,970.545501,239.076972,0.246333
4,sub-010079,EC,1004.547026,188.642101,0.187788


In [7]:
df_global[GLOBAL_FEATURES].describe()

,edges_mean,edges_std,edges_cv
count,108.000000,108.000000,108.000000
mean,977.548678,205.687255,0.211708
std,160.458025,35.675409,0.026203
min,599.400610,116.512176,0.161313
25%,860.381291,180.122321,0.192693
50%,983.351296,202.513614,0.209107
75%,1087.483859,230.470539,0.229020
max,1382.880020,283.987335,0.277659


In [8]:
df_global.groupby("condition")[GLOBAL_FEATURES].describe()

edges_mean                                                  \
               count        mean         std         min         25%   
condition                                                              
EC              54.0  992.409545  146.404108  634.610574  911.502415   
EO              54.0  962.687812  173.479558  599.400610  841.816345   

                                                edges_std              ...  \
                  50%          75%          max     count        mean  ...   
condition                                                              ...   
EC         993.802237  1089.372013  1343.476868      54.0  203.954727  ...   
EO         980.038892  1083.447128  1382.880020      54.0  207.419783  ...   

                                  edges_cv                                \
                  75%         max    count      mean       std       min   
condition                                                                  
EC         229.136466  283.987335     54.0  0.206389  0.025373  0.161313   
EO         235.503082  276.975855     54.0  0.217027  0.026167  0.169520   

                                                   
                25%       50%       75%       max  
condition                                          
EC         0.189406  0.203361  0.220908  0.266758  
EO         0.198855  0.215745  0.236511  0.277659  

[2 rows x 24 columns]

### Boxplots EO vs EC

Número médio de arestas

In [9]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_mean",
    color="condition",
    points="all",
    title="Número Médio de Arestas por Condição"
)

fig.show()

### Variabilidade da conectividade

In [10]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_cv",
    color="condition",
    points="all",
    title="Coeficiente de Variação da Conectividade ao Longo do Tempo"
)
fig.show()

### Correlação entre métricas globais

In [11]:
corr = df_global[GLOBAL_FEATURES].corr()
corr

,edges_mean,edges_std,edges_cv
edges_mean,1.000000,0.721839,-0.304291
edges_std,0.721839,1.000000,0.432579
edges_cv,-0.304291,0.432579,1.000000


In [12]:
fig = px.imshow(
    corr,
    text_auto=True,
    title="Matriz de Correlação entre Métricas Globais",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)
fig.show()